[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/wing/wing.ipynb)

# Lift without vorticity

A wing in a steady stream: the air over it moves faster than the air under it, and the difference in pressure lifts it. The velocity is the derivative of a potential field with a scalar part and a bivector part. The potential's gradient at each point is a map from a small step to the change in the potential, and contracting that map against an open vector is the derivative: of the potential's reverse, twice the velocity; of the potential itself, zero everywhere, so the flow has no vorticity anywhere. Yet the wing is lifted by a circulation around it.

In the notation of complex analysis this is the classical route of Joukowski: the complex potential of the flow past a cylinder, carried to the wing by his map $z = \zeta + c^2/\zeta$, with the flow analytic by the equations of Cauchy and Riemann, and the lift given by the Kutta–Joukowski theorem.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext, concatenate, stack
from examples.mechanics.wing import render
from examples.mechanics.wing.core import Flow, Wing, ga, pitched, rings

np.set_printoptions(precision=3, suppress=True)

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Vector = ga.gatype.vector()
Bivector = ga.gatype.bivector()
Even = ga.gatype.even()
PotentialGradient = ga.gatype((Even, Vector))                                # Even <- Vector

# The wing level along the stream; the cylinder's centre before the map sets its thickness and camber.
thickness, camber, scale = 0.1, 0.08, 1.0
speed, density = 1.0, 1.0
attack = np.deg2rad(8.0)
rings_count, angles, reach = 140, 360, 6.0
# A probe point ahead of the leading edge, in steps of the cylinder's radius from its centre.
probe_reach = 1.3
# The grid of circles drawn: centres of the circles through both critical points, across the chord
# in steps of the critical point's distance, and distance ratios of the circles around one of them;
# points along each circle; and the frames of the cylinder's deformation into the wing.
through_offsets = np.array([-1.5, -0.6, 0.0, 0.6, 1.5])
distance_ratios = np.array([0.2, 0.45, 0.7])
outline_points = 400
morph_frames, morph_duration_ms = 24, 80

## 1. The potential of the flow past a cylinder

The velocity is a vector field, and it is the derivative of a potential with two parts: a scalar, the velocity potential, which rises along the flow, and a bivector, the stream function, which rises across it and measures the fluid passing between two streamlines. Together they make one multivector of even grade, `W`. For a uniform stream `U` it is the geometric product of the stream and the position, `U * r`. The cylinder adds the stream's image in it: the point inverted in the cylinder, `inverted = radius**2 * r.inverse()`, at the same direction but the radius squared over the distance, gives the image potential `inverted * U`, the stream's own potential reversed and seen from the inverted point. A circulation, itself a bivector, adds a swirl around the cylinder. Below is the potential's gradient at each point, a map from a small step to the change in `W`, `Even <- Vector`, and the stream function as a bivector.

In the notation of complex analysis, the even multivectors of the plane read as complex numbers, with `xy` as the imaginary unit and a chosen direction as the real axis: a point `p` as that direction times `p`, `W` as the complex potential $\phi + i\psi$, and the image term as Milne-Thomson's circle theorem, $W(\zeta) = f(\zeta) + \overline{f(R^2/\bar\zeta)}$.

In [ ]:
def cylinder(centre: Vector, edge: Vector, swirl: Bivector, stream: Vector, plane: Vector) -> tuple[PotentialGradient, Bivector]:
    """The potential's gradient past a cylinder through `centre + edge`, and the stream function."""
    # Each point's position relative to the cylinder's centre, and the cylinder's radius squared.
    offsets = plane - centre                                                   # [...] Vector
    radius_squared = edge | edge                                               # [] Scalar
    # Each point inverted in the cylinder: the same direction, at the radius squared over its distance.
    inverted = radius_squared * offsets.inverse()                             # [...] Vector
    # The uniform stream's potential `stream * r` changes by `stream * step` for a step, the same everywhere.
    uniform = stream * Vector                                                  # [] Even <- Vector
    # The image's potential is the stream's reversed at the inverted point, `inverted * stream`. A step
    # moves the inverted point by minus the step reflected in it, the sandwich `inverted >> step`,
    # over the radius squared. It cancels the stream through the cylinder's surface.
    image = -(inverted >> Vector) * stream / radius_squared                    # [...] Even <- Vector
    # The swirl's potential changes by the circulation times `r.inverse() * step`, over two pi.
    turning = -swirl * offsets.inverse() * Vector / (2 * np.pi)                # [...] Even <- Vector
    # The stream function: the bivector parts of the stream's and the image's potentials, and the
    # swirl's, which grows with the logarithm of the distance.
    flux = (stream ^ offsets) + (inverted ^ stream) - swirl * (offsets | offsets).log() / (4 * np.pi)
    return uniform + image + turning, flux

## 2. The circulation

The wing's trailing edge is sharp, and the flow can only leave it smoothly if it stands still there before the map. One circulation makes it so, the Kutta condition, and it comes out as a bivector: the stream wedged with the direction from the cylinder's centre to the trailing edge, `4 * pi * (stream ^ edge)`. Its lift is the circulation contracted with the stream, times the density.

In scalar notation the lift reads as $L = \rho U \Gamma$, with $\Gamma = 4\pi U R \sin(\alpha + \beta)$ for a cylinder of radius $R$ whose trailing edge lies at the angle $\beta$ below its centre.

In [ ]:
# The wing level along the stream, then pitched nose up by the angle of attack: the rotor turns the
# cylinder's centre and the chord together, and the map built on the chord turns with them.
wing = pitched(Wing(-thickness * mv.x + camber * mv.y, scale, mv.x), attack)   # [] Wing
stream = speed * mv.x                                                          # [] Vector
# From the cylinder's centre to the trailing edge, where the flow must stand still.
edge = wing.scale * wing.chord - wing.centre                                   # [] Vector
# The circulation that stops it there, and the lift it gives: a bivector, the plane's orientation
# times the strength, signed by which side of the stream the edge lies on.
swirl = 4 * np.pi * (stream ^ edge)                                            # [] Bivector
lift = density * (swirl | stream)                                              # [] Vector

## 3. Building the map

The Joukowski map turns the cylinder into the wing. We build it around two points on the chord, `critical` and `-critical`. Every point can be described by how it sees them: the angle at which it sees the segment between them, and the ratio of its distances to them, its bipolar coordinates. One even multivector holds both, the ratio of the point's vectors to the two, `(p - critical) * (p + critical).inverse()`, which turns and scales the one into the other. The map doubles both by squaring that ratio, and moves the two points twice as far out: the image is the point that sees `2 * critical` and `-2 * critical` at the squared ratio. Written out, this comes to `p + (critical >> p.inverse())`.

The points seeing the segment at a fixed angle lie on a circle through both critical points, and the points at a fixed distance ratio on a circle around one of them; together they make a grid of circles. The map sends this grid onto the same kind of grid around the moved points, every circle onto a circle, only with its angle or ratio doubled. The circle centred between the critical points sees them at a right angle, which doubles to a straight one: it flattens onto the plate between the moved points. The wing's circle cuts across the grid: it passes through `critical` and goes around `-critical`. At `critical`, where the doubling folds the grid, its smooth edge becomes a cusp, the sharp trailing edge.

In the notation of complex analysis the construction reads as $\frac{z - 2c}{z + 2c} = \left(\frac{\zeta - c}{\zeta + c}\right)^2$, which solves to $z = \zeta + c^2/\zeta$.

In [ ]:
def joukowski(critical: Vector, plane: Vector) -> Vector:
    """The image of each point: the point seeing twice the critical points at the squared ratio."""
    # How the point sees the critical points: the ratio of its vectors to them, an even multivector
    # whose angle is the angle it sees the segment between them at, and whose size is the ratio of
    # its distances to them.
    seen = (plane - critical) * (plane + critical).inverse()                   # [...] Even
    # Squaring an even multivector of the plane doubles its angle and squares its size.
    doubled = seen * seen                                                      # [...] Even
    # The point whose vectors to the moved critical points have that ratio: solving
    # `image - 2 * critical == doubled * (image + 2 * critical)` for the image.
    return (1 - doubled).inverse() * (1 + doubled) * (2 * critical)            # [...] Vector


# The trailing edge is a critical point: the wing's circle passes through it.
critical = wing.scale * wing.chord                                             # [] Vector
turns = (mv.xy * np.linspace(0, np.pi, outline_points)).exp()                 # [outline_points] Rotor
# Circles through both critical points: their centres lie across the chord, through the map's
# centre, and every point on one sees the segment at the same angle, the inscribed angle.
across = through_offsets * critical.dual()                                     # [through] Vector
reach_through = ((critical | critical) + (across | across)).square_root()      # [through] Scalar
through = across[:, None] + reach_through[:, None] * (turns >> wing.chord)[None, :]   # [through, outline_points] Vector
# Circles around each critical point, the points with one ratio of distances to the two: their
# centres lie along the chord, beyond the critical point they surround.
ratios = np.concatenate([distance_ratios, 1 / distance_ratios])
centres = critical * ((1 + ratios**2) / (1 - ratios**2))                       # [around] Vector
reach_around = (critical | critical).square_root() * (2 * ratios / np.abs(1 - ratios**2))   # [around] Scalar
around = centres[:, None] + reach_around[:, None] * (turns >> wing.chord)[None, :]   # [around, outline_points] Vector
grid = concatenate([through, around])                                          # [circles, outline_points] Vector
wing_circle = wing.centre + (turns >> edge)                                    # [outline_points] Vector
render.bipolar(grid, joukowski(critical, grid), wing_circle, joukowski(critical, wing_circle), critical);

## 4. Carried to the wing

A small step moves under the map by its Jacobian, `Vector <- Vector`, written out from `p + (critical >> p.inverse())`: the step, less the step turned and scaled by the even multivector `critical * p.inverse()`, which is why the map keeps angles. The potential is the same at a point and at its image, so composing its gradient with the inverse of the Jacobian, `jacobian.solve(1 * Vector)`, carries it to the wing. The velocity is half the derivative of the potential's reverse, a contraction against an open vector. In the picture the pressure is low where the flow is fast, over the wing, and the arrow is the lift.

In [ ]:
def flow(wing: Wing, swirl: Bivector, stream: Vector, plane: Vector, critical: Vector) -> Flow:
    # The flow past the cylinder, at points around it; the map then only moves where things are.
    gradient, flux = cylinder(wing.centre, wing.scale * wing.chord - wing.centre, swirl, stream, plane)
    # Where the map sends each point.
    points = joukowski(critical, plane)                                        # [...] Vector
    # How the map moves a small step at each point: the step, less the step turned and scaled by the
    # even `critical * p.inverse()`, from the inverse changing by `-p.inverse() >> step`, as in the image.
    jacobian = Vector - ((critical * plane.inverse()) >> Vector)              # [...] Vector <- Vector
    # The potential at an image is the potential at the point, so a step at the image is first
    # undone by the Jacobian, then fed to the cylinder's gradient.
    carried = gradient(jacobian.solve(1 * Vector))                             # [...] Even <- Vector
    # The velocity: half the derivative of the potential's reverse. The reverse flips the sign of the
    # stream function, which turns the derivative's zero into twice the velocity potential's gradient.
    velocity = 0.5 * (Vector * carried(Vector).reverse()).contract()          # [...] Vector
    return Flow(points, velocity, carried, flux)


# Points on rings around the cylinder, from its surface out; their images surround the wing.
plane = rings(wing, rings_count, angles, reach)                                # [rings, angles] Vector
flows = flow(wing, swirl, stream, plane, critical)                                       # [rings, angles] Flow
render.figure(flows, lift, speed);

Moving the critical points apart deforms the cylinder into the wing. While they lie inside it, the map is smooth on its surface and the body is round; as `critical` reaches the surface, the doubled angle there turns into the cusp. The yellow points are their images, `2 * critical` and its negative. They start together at the centre of the map, while the cylinder's centre before the map, the white point, sits behind it by the thickness and above it by the camber: that offset is all that makes the wing differ front to back and top to bottom. The circulation is held at the wing's throughout, so the last frame is the wing's flow.

In [ ]:
# The critical points moved apart from the map's centre out to the trailing edge, and the flow at
# each step. The cylinder and its circulation stay put; only the map changes, the identity at first.
fractions = np.linspace(0.0, 1.0, morph_frames)
# Marked: where the map sends the critical points, twice as far out, and the cylinder's centre.
frames = [render.marked_frame(flow(wing, swirl, stream, plane, fraction * critical), lift, speed,
                              2 * fraction * stack([critical, -critical]), wing.centre)
          for fraction in fractions]
render.inline(frames, morph_duration_ms)

## 5. The derivative of the potential

Contracting the potential's gradient against an open vector, `(Vector * gradient(Vector)).contract()`, gives its derivative, and it vanishes everywhere: the flow has neither divergence nor vorticity. This is the Cauchy–Riemann condition.

In the notation of complex analysis the vanishing derivative reads as the Cauchy–Riemann equations for $\phi + i\psi$, the Joukowski map as $z = \zeta + c^2/\zeta$ with the chord as the real axis, and the velocity as the conjugate of $dW/dz$.

In [ ]:
# The potential's derivative at every point: the open vector paired with the gradient's step.
derivatives = (Vector * flows.potential_gradient(Vector)).contract()          # [rings, angles] Even
# A point a little ahead of the leading edge, opposite the trailing edge across the cylinder.
probe = flow(wing, swirl, stream, wing.centre - probe_reach * edge, critical)            # [] Flow

Printed below, the gradient ahead of the leading edge. Its scalar row is the velocity there: the potential rises along the flow. Its `xy` row is the same velocity turned a quarter turn: the stream function rises across the flow. That one row is the other turned is the Cauchy–Riemann equations, which the contraction checks at every point at once.

In [ ]:
print(f"potential gradient ahead of the leading edge\n{probe.potential_gradient.coefficient_table()}\n")
print("velocity there:", probe.velocity.kernel)
print("largest derivative anywhere:", np.abs(derivatives.kernel).max())

## 6. Lift

The pressure on the surface pushes the wing by half the density times the squared speed, along the inward normal at each step around it. Summed around the wing, that force is the lift from the circulation, across the stream, and nothing along it: an ideal flow that lifts but does not drag, d'Alembert's paradox. The circulation lives in the loop around the wing, not at any point of the flow.

In [ ]:
# The surface: the innermost ring, with each point's step to the next one around.
surface = flow(wing, swirl, stream, rings(wing, 1, angles, 1.0)[0], critical)           # [angles] Flow
next_idx = (np.arange(angles) + 1) % angles
steps = surface.points[next_idx] - surface.points                              # [angles] Vector
# The pressure is low where the flow is fast: it pushes by half the density times the squared speed,
# averaged over each step, along the inward normal, the step turned a quarter turn back.
speed_squared = surface.velocity | surface.velocity                            # [angles] Scalar
force = (0.5 * density * (speed_squared + speed_squared[next_idx]) / 2 * -steps.dual()).sum(axis=-1)   # [] Vector
print(f"pressure force, {force.output_subspace}: {force.kernel}")
print(f"lift from the circulation, {lift.output_subspace}: {lift.kernel}")

In [ ]:
# checks
# The map built in three steps is `p + (critical >> p.inverse())`.
np.testing.assert_allclose((joukowski(critical, plane) - (plane + (critical >> plane.inverse()))).kernel, 0.0, atol=1e-12)
# A circle around a critical point lands on a circle around the moved one, its distance ratio squared.
landed = joukowski(critical, around)                                           # [around, outline_points] Vector
seen_after = (landed - 2 * critical) * (landed + 2 * critical).inverse()       # [around, outline_points] Even
np.testing.assert_allclose(seen_after.scalar_norm_squared().kernel[..., 0], (ratios**4)[:, None] * np.ones(outline_points), rtol=1e-9)
# The potential has no derivative anywhere.
np.testing.assert_allclose(derivatives.kernel, 0.0, atol=1e-9)
# The stream function changes across a small step by the velocity's flux through it.
step = mv.vector([0.3, -0.2]) * 1e-6                                           # [] Vector
forward, backward = flow(wing, swirl, stream, plane + step, critical), flow(wing, swirl, stream, plane - step, critical)
crossing = (forward.points - backward.points) / 2                              # [rings, angles] Vector
np.testing.assert_allclose(((forward.stream - backward.stream) / 2 - (flows.velocity ^ crossing)).kernel, 0.0, atol=1e-12)
# The pressure force is the lift, to the resolution of the surface.
np.testing.assert_allclose((force - lift).kernel, 0.0, atol=1e-3)